# Milestone 4 Lab, notebook 2 of 7: Run records

## Where we are

| Notebook | What you do |
|---|---|
| 1. A figure, traced by hand | Find every record behind one figure yourself, and add them up |
| **2. Run records** (you are here) | Give every pipeline run an id and a record, and keep every run |
| 3. Lineage | Save which records made each figure, and prove it adds up |
| 4. Two runs compared | Find what changed between two runs: settings, inputs, figures, records |
| 5. A trace of the analyst | Give the analyst the new tools, and record every step it takes |
| 6. Scoring an agent | Measure an agent with a confusion matrix, kappa, and the cost of mistakes |
| 7. Two experiments | Run two versions of the analyst on the same questions, and compare the scores |

Model requests in this notebook: **0**. A full run of this lab costs about 5 US cents.

## What this notebook is for

In notebook 1 you found two values for the same figure, and no record of why. The cause is simple: each time the pipeline runs, it overwrites what the last run wrote.

Here you fix that. Every run gets an **id** and a **record**, and keeps its tables in its own folder. Nothing is overwritten.

No AI is used in this notebook. A run record must be exact, so code writes it.

In [ ]:
import hashlib
import shutil

import pandas as pd
from lab_helpers import LAB_WAREHOUSE, SOURCE, fingerprint_by_hand, read_run_table, show_doc

## Step 1: The words

Four new words are used in this milestone. The first two matter for this notebook.

In [ ]:
show_doc("BRD.md", "5. Words used in this milestone")

## Step 2: A fingerprint, by hand

A run record should say **which inputs** the run used. Copying every input file into the record would be wasteful. We store a **fingerprint**: a short code worked out from a file's content.

The standard tool is a hash function called SHA-256. Give it any content, and it returns 64 characters. Try it on two words that differ by one letter.

In [ ]:
print(hashlib.sha256(b"refund").hexdigest())
print(hashlib.sha256(b"refunds").hexdigest())

One extra letter, and the whole code is different. That is what makes it useful: if two fingerprints are equal, the contents are equal.

64 characters is more than we need to read. We keep the first 12.

In [ ]:
costs_file = SOURCE / "product_costs.csv"

content = costs_file.read_bytes()
full_hash = hashlib.sha256(content).hexdigest()
fingerprint = full_hash[:12]

print("File:       ", costs_file.name, f"({len(content)} bytes)")
print("Fingerprint:", fingerprint)

Now change one character in a **copy** of the file and fingerprint the copy. We change one cost from `18.00` to `18.01`.

In [ ]:
LAB_WAREHOUSE.mkdir(exist_ok=True)
changed_file = LAB_WAREHOUSE / "product_costs_changed.csv"

text = costs_file.read_text(encoding="utf-8")
changed_text = text.replace("18.00", "18.01", 1)
changed_file.write_text(changed_text, encoding="utf-8")

print("Original:", fingerprint_by_hand(costs_file))
print("Changed: ", fingerprint_by_hand(changed_file))

changed_file.unlink()

`fingerprint_by_hand` is the three lines you just ran, kept in `lab_helpers.py`.

## Step 3: What a run should record

Here is the design, from the **LLD**.

In [ ]:
show_doc("LLD.md", "3. Run records")

The record answers four questions about a run:

- **Which run?** `run_id`, `started_at`, `note`
- **With which choices?** `settings`
- **From what?** `inputs` (a fingerprint for each source table) and `code` (a fingerprint for each calculation)
- **What happened?** the check outcomes, the findings, the quarantined rows, and whether the lineage adds up

## Step 4: Make the first run

The production pipeline now writes this record. We run it into a lab folder, `notebooks/lab_warehouse/`, so your experiments stay separate from the real history. The cell clears that folder first, so you can run this notebook again.

The first run uses `quarantine=False`. That counts every record, copies included: the way the business calculated before milestone 3.

In [ ]:
from retail_profit_stack.transform.pipeline import run_pipeline

if LAB_WAREHOUSE.exists():
    shutil.rmtree(LAB_WAREHOUSE)

first_run = run_pipeline(
    "sql",
    warehouse_dir=LAB_WAREHOUSE,
    quarantine=False,
    note="Duplicated records are counted, as before milestone 3",
)

print("This run's id:", first_run.run_id)

Here is its record. Read it against the four questions above.

In [ ]:
first_run.record

## Step 5: Make the second run

Now the way milestone 3 calculates: duplicated records are set aside. Only one argument differs.

In [ ]:
second_run = run_pipeline(
    "sql",
    warehouse_dir=LAB_WAREHOUSE,
    quarantine=True,
    note="Duplicated records are set aside",
)

print("This run's id:", second_run.run_id)
print("Rows set aside:", second_run.record["quarantined"])

## Step 6: Read the history

The history is one file, `run_history.json`, with one entry per run.

In [ ]:
from retail_profit_stack.lineage.runs import read_history

history = read_history(LAB_WAREHOUSE)

rows = []
for record in history:
    rows.append({
        "run_id": record["run_id"],
        "started_at": record["started_at"],
        "note": record["note"],
        "quarantine": record["settings"]["quarantine"],
        "findings": record["findings"],
        "lineage_adds_up": record["lineage_adds_up"],
    })

pd.DataFrame(rows)

Each run also has its own folder of tables. The second run did not touch the first run's folder.

In [ ]:
for folder in sorted((LAB_WAREHOUSE / "runs").iterdir()):
    files = sorted(file.name for file in folder.iterdir())
    print(folder.name, "holds", len(files), "tables")

print()
print(files)

So both values of the trail water bottle's figure now exist at the same time. This is what was missing in notebook 1.

In [ ]:
for run_id in ["RUN-0001", "RUN-0002"]:
    profit = read_run_table(run_id, "monthly_product_profit")
    bottle = profit[(profit["product_id"] == "P001") & (profit["month"] == "2026-10")]
    print(run_id, bottle["contribution_profit"].iloc[0])

## Step 7: Run again with nothing changed

What should a third run look like if nothing changed? The same inputs, the same code, and the same settings should give the same figures.

In [ ]:
third_run = run_pipeline(
    "sql", warehouse_dir=LAB_WAREHOUSE, quarantine=True, note="Run again, nothing changed"
)

second = second_run.record
third = third_run.record

for field in second:
    same = "same" if second[field] == third[field] else "DIFFERENT"
    print(f"{field:18} {same}")

Only fields that describe the run itself can differ: its id, its note, when it started, and how long it took. Every fingerprint is the same, and so is every result.

This is the rule that makes run history useful: **if the fingerprints and settings are equal, the figures are equal.** So when a figure changes, one of them must have changed, and the record shows which.

In [ ]:
second_profit = read_run_table("RUN-0002", "monthly_product_profit")
third_profit = read_run_table("RUN-0003", "monthly_product_profit")

print("The two profit tables are equal:", second_profit.equals(third_profit))

## What we now know

- A **fingerprint** is a short code worked out from content. Equal fingerprints mean equal content.
- A **run record** says which run, with which settings, from which inputs and code, and what happened.
- Every run keeps its own folder, so an old figure is never lost. `RUN-0001` still says $5,188.20 and `RUN-0002` says $5,156.94.
- Equal fingerprints and settings give equal figures. When a figure moves, the record shows what moved first.

In production, `uv run python -m retail_profit_stack.make_history` makes these first two runs in the real warehouse folder.

**Next:** notebook 3. A run now records what went in. Lineage records which of those records made each figure.